# Group-based image split files

Purpose: Create the final group-based image train/validation/test split using group_key so that related document samples do not appear across splits.

In [1]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

project_root = Path.cwd().parent
processed_dir = project_root / "data" / "processed"

print("Project root:", project_root)
print("Processed dir:", processed_dir)

Project root: c:\Users\Admin\Desktop\thesis\multimodal-fraud-detection-thesis
Processed dir: c:\Users\Admin\Desktop\thesis\multimodal-fraud-detection-thesis\data\processed


In [2]:
full_df = pd.read_csv(processed_dir / "image_dataset_final.csv").copy()

old_train = pd.read_csv(processed_dir / "image_train.csv").copy()
old_val = pd.read_csv(processed_dir / "image_val.csv").copy()
old_test = pd.read_csv(processed_dir / "image_test.csv").copy()

print("FULL:", full_df.shape)
print("OLD TRAIN:", old_train.shape)
print("OLD VAL:", old_val.shape)
print("OLD TEST:", old_test.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\Admin\\Desktop\\thesis\\multimodal-fraud-detection-thesis\\data\\processed\\image_train.csv'

In [3]:
old_train_groups = set(old_train["group_key"])
old_val_groups = set(old_val["group_key"])
old_test_groups = set(old_test["group_key"])

print("OLD Train-Val overlap:", len(old_train_groups & old_val_groups))
print("OLD Train-Test overlap:", len(old_train_groups & old_test_groups))
print("OLD Val-Test overlap:", len(old_val_groups & old_test_groups))

NameError: name 'old_train' is not defined

In [ ]:
group_df = full_df[["group_key"]].drop_duplicates().copy()

train_groups, temp_groups = train_test_split(
    group_df["group_key"],
    test_size=0.30,
    random_state=42
)

val_groups, test_groups = train_test_split(
    temp_groups,
    test_size=0.50,
    random_state=42
)

new_train = full_df[full_df["group_key"].isin(train_groups)].copy()
new_val = full_df[full_df["group_key"].isin(val_groups)].copy()
new_test = full_df[full_df["group_key"].isin(test_groups)].copy()

In [ ]:
new_train_groups = set(new_train["group_key"])
new_val_groups = set(new_val["group_key"])
new_test_groups = set(new_test["group_key"])

print("NEW Train-Val overlap:", len(new_train_groups & new_val_groups))
print("NEW Train-Test overlap:", len(new_train_groups & new_test_groups))
print("NEW Val-Test overlap:", len(new_val_groups & new_test_groups))

NEW Train-Val overlap: 0
NEW Train-Test overlap: 0
NEW Val-Test overlap: 0


In [ ]:
print("OLD SPLITS")
for name, df in [("TRAIN", old_train), ("VAL", old_val), ("TEST", old_test)]:
    print(f"{name}: rows={len(df)}, groups={df['group_key'].nunique()}")

print("\nNEW SPLITS")
for name, df in [("TRAIN", new_train), ("VAL", new_val), ("TEST", new_test)]:
    print(f"{name}: rows={len(df)}, groups={df['group_key'].nunique()}")

OLD SPLITS
TRAIN: rows=6320, groups=915
VAL: rows=1419, groups=296
TEST: rows=2425, groups=613

NEW SPLITS
TRAIN: rows=7182, groups=953
VAL: rows=1508, groups=204
TEST: rows=1474, groups=205


In [ ]:
print("OLD CLASS BALANCE")
for name, df in [("TRAIN", old_train), ("VAL", old_val), ("TEST", old_test)]:
    print(f"\n{name}")
    print(df["image_class"].value_counts())
    print(df["image_class"].value_counts(normalize=True))

print("\nNEW CLASS BALANCE")
for name, df in [("TRAIN", new_train), ("VAL", new_val), ("TEST", new_test)]:
    print(f"\n{name}")
    print(df["image_class"].value_counts())
    print(df["image_class"].value_counts(normalize=True))

OLD CLASS BALANCE

TRAIN
image_class
bona_fide    3300
forged       3020
Name: count, dtype: int64
image_class
bona_fide    0.522152
forged       0.477848
Name: proportion, dtype: float64

VAL
image_class
bona_fide    729
forged       690
Name: count, dtype: int64
image_class
bona_fide    0.513742
forged       0.486258
Name: proportion, dtype: float64

TEST
image_class
forged       1521
bona_fide     904
Name: count, dtype: int64
image_class
forged       0.627216
bona_fide    0.372784
Name: proportion, dtype: float64

NEW CLASS BALANCE

TRAIN
image_class
forged       3731
bona_fide    3451
Name: count, dtype: int64
image_class
forged       0.519493
bona_fide    0.480507
Name: proportion, dtype: float64

VAL
image_class
forged       765
bona_fide    743
Name: count, dtype: int64
image_class
forged       0.507294
bona_fide    0.492706
Name: proportion, dtype: float64

TEST
image_class
bona_fide    739
forged       735
Name: count, dtype: int64
image_class
bona_fide    0.501357
forged    

In [ ]:
print("NEW SOURCE DATASET BALANCE")
for name, df in [("TRAIN", new_train), ("VAL", new_val), ("TEST", new_test)]:
    print(f"\n{name}")
    print(df["source_dataset"].value_counts())
    print(df["source_dataset"].value_counts(normalize=True))

NEW SOURCE DATASET BALANCE

TRAIN
source_dataset
midv         2788
fantasyid    2354
fmidv        2040
Name: count, dtype: int64
source_dataset
midv         0.388193
fantasyid    0.327764
fmidv        0.284043
Name: proportion, dtype: float64

VAL
source_dataset
midv         596
fantasyid    492
fmidv        420
Name: count, dtype: int64
source_dataset
midv         0.395225
fantasyid    0.326260
fmidv        0.278515
Name: proportion, dtype: float64

TEST
source_dataset
midv         616
fantasyid    438
fmidv        420
Name: count, dtype: int64
source_dataset
midv         0.417910
fantasyid    0.297151
fmidv        0.284939
Name: proportion, dtype: float64


In [ ]:
new_train["split"] = "train"
new_val["split"] = "val"
new_test["split"] = "test"

new_image_all = pd.concat([new_train, new_val, new_test], ignore_index=True)

print(new_image_all.shape)
print(new_image_all["split"].value_counts())

(10164, 10)
split
train    7182
val      1508
test     1474
Name: count, dtype: int64


In [ ]:
new_train.to_csv(processed_dir / "image_train_group_test.csv", index=False)
new_val.to_csv(processed_dir / "image_val_group_test.csv", index=False)
new_test.to_csv(processed_dir / "image_test_group_test.csv", index=False)

print("Saved test image split files.")

Saved test image split files.


In [ ]:
print((processed_dir / "image_train_group_test.csv").exists())
print((processed_dir / "image_val_group_test.csv").exists())
print((processed_dir / "image_test_group_test.csv").exists())


True
True
True
